<a href="https://colab.research.google.com/github/yuhui-0611/ESAA/blob/main/ESAA_OB_summer_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data)

# 주문 단위 Baseline Dataset

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
customers = pd.read_csv('/content/drive/MyDrive/ESAA_OB_pj/customer/olist_customers_dataset.csv')
orders = pd.read_csv('/content/drive/MyDrive/ESAA_OB_pj/customer/olist_orders_dataset.csv')
items = pd.read_csv('/content/drive/MyDrive/ESAA_OB_pj/customer/olist_order_items_dataset.csv')
payments = pd.read_csv('/content/drive/MyDrive/ESAA_OB_pj/customer/olist_order_payments_dataset.csv')

In [ ]:
# 1. 데이터 크기와 컬럼 확인
print("customers:", customers.shape)
print("orders:", orders.shape)
print("items:", items.shape)
print("payments:", payments.shape)

print(customers.columns)
print(orders.columns)
print(items.columns)
print(payments.columns)

customers: (99441, 5)
orders: (99441, 8)
items: (112650, 7)
payments: (103886, 5)
Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')
Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='object')
Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='object')


- customers와 orders는 둘 다 customer_id가 있음 → customer_id 기준 병합
- orders, items, payments는 모두 order_id가 있음 → order_id 기준 병합
- items, payments는 주문 1건에 여러 행이 있을 수 있음 → 바로 병합하면 행이 불필요하게 늘어날 수 있어서 먼저 order_id별 집계 후 병합
  - 예를 들어 어떤 order_id가 payments에 2줄 있으면, 그냥 병합했을 때 base에서도 그 주문이 2줄로 늘어남
  - 그러면 고객별 구매 횟수나 금액 계산이 꼬일 수 있음

## **base = orders + customers**

In [ ]:
# 2. orders + customers 병합
base = pd.merge(
    orders,
    customers,
    on='customer_id',
    how='left'
)

print(base.shape)
base.head()

(99441, 12)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [ ]:
base.isnull().sum()

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0
customer_unique_id,0
customer_zip_code_prefix,0


< null 설명 >
- order_approved_at: 승인 안 된 주문 또는 누락
- order_delivered_carrier_date: 배송사 전달 전 취소/미배송
- order_delivered_customer_date: 고객에게 배송 완료되지 않은 주문

In [ ]:
# 3. 주문별 결제 정보 집계
payment_agg = payments.groupby('order_id', as_index=False).agg(
    payment_value=('payment_value', 'sum'),
    payment_count=('payment_sequential', 'count'),
    payment_installments_mean=('payment_installments', 'mean'),
    payment_type_nunique=('payment_type', 'nunique')
)

print(payment_agg.shape)
payment_agg.head()

(99440, 5)


,order_id,payment_value,payment_count,payment_installments_mean,payment_type_nunique
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2.0,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3.0,1
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3.0,1


- 원래 payments는 103886행이었는데, order_id별로 묶어서 99440행이 됨
- 즉, 여러 결제 행이 있던 주문들이 주문 단위로 잘 합쳐짐

## **+ payment**

In [ ]:
# 4. base + payment_agg 병합
base = pd.merge(
    base,
    payment_agg,
    on='order_id',
    how='left'
)

print(base.shape)
base.head()

(99441, 16)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_value,payment_count,payment_installments_mean,payment_type_nunique
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,3.0,1.0,2.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,1.0,1.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,1.0,3.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,1.0,1.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,1.0,1.0,1.0


- base의 행 수가 99441 그대로 유지됨
- 행 수가 늘어나지 않았으니까 병합이 잘 된 것

In [ ]:
base[['payment_value', 'payment_count',
      'payment_installments_mean', 'payment_type_nunique']].isnull().sum()

,0
payment_value,1
payment_count,1
payment_installments_mean,1
payment_type_nunique,1


- orders에는 있는데 payments에는 없는 주문이 1개 있다는 뜻

In [ ]:
# 5. 주문별 상품 정보 집계
items_agg = items.groupby('order_id', as_index=False).agg(
    item_count=('order_item_id', 'count'),
    product_count=('product_id', 'nunique'),
    seller_count=('seller_id', 'nunique'),
    product_price_sum=('price', 'sum'),
    freight_value_sum=('freight_value', 'sum')
)

print(items_agg.shape)
items_agg.head()

(98666, 6)


,order_id,item_count,product_count,seller_count,product_price_sum,freight_value_sum
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14


## **+ items**

In [ ]:
# 6. base + items_agg 병합
base = pd.merge(
    base,
    items_agg,
    on='order_id',
    how='left'
)

print(base.shape)
base.head()

(99441, 21)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,customer_state,payment_value,payment_count,payment_installments_mean,payment_type_nunique,item_count,product_count,seller_count,product_price_sum,freight_value_sum
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,SP,38.71,3.0,1.0,2.0,1.0,1.0,1.0,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,BA,141.46,1.0,1.0,1.0,1.0,1.0,1.0,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,GO,179.12,1.0,3.0,1.0,1.0,1.0,1.0,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,RN,72.20,1.0,1.0,1.0,1.0,1.0,1.0,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,SP,28.62,1.0,1.0,1.0,1.0,1.0,1.0,19.90,8.72


- 기존 16개에서 items 관련 5개가 추가돼서 21개가 나옴

In [ ]:
base[['item_count', 'product_count', 'seller_count',
      'product_price_sum', 'freight_value_sum']].isnull().sum()

,0
item_count,775
product_count,775
seller_count,775
product_price_sum,775
freight_value_sum,775


- orders에는 있는데 items에는 없는 주문이 775개 있다는 뜻
- 취소, unavailable, 미완료 주문 등이 여기에 포함될 수 있음

### datetime 변환

- for "Recency와 Tenure 계산"
  - Recency = 최근 구매 후 얼마나 지났는지
    - Recency가 작을수록 최근에 구매한 고객
    - ```
      recency = 기준일 - 고객의 마지막 구매일
      ```
  - Tenure = 첫 구매부터 마지막 구매까지 고객이 활동한 기간
    - Tenure가 길수록 오래 활동한 고객
    - ```
      tenure = 고객의 마지막 구매일 - 고객의 첫 구매일
      ```

In [ ]:
# 7. 날짜 컬럼 datetime 변환

date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    base[col] = pd.to_datetime(base[col], errors='coerce')

base[date_cols].dtypes

,0
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


In [ ]:
base[date_cols].isnull().sum()

,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


In [ ]:
base['order_status'].value_counts()

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


- 주문 상태를 보면 대부분이 delivered
- LTV/RFM용 고객 데이터는 보통 실제로 구매가 완료된 주문만 사용

In [ ]:
base.to_csv(
    '/content/drive/MyDrive/ESAA_OB_pj/customer/order_baseline.csv',
    index=False
)

# 고객 단위 Dataset

- 현재 base = 주문별 결제 이력과 날짜가 담긴 테이블
- base를 필터링하고 groupby → 고객 단위 customer_features 생성

In [ ]:
# 8. 배송 완료 주문만 필터링
base_delivered = base[base['order_status'] == 'delivered'].copy()

print(base_delivered.shape)
base_delivered[['order_status', 'payment_value', 'item_count']].isnull().sum()

(96478, 21)


,0
order_status,0
payment_value,1
item_count,0


In [ ]:
# 9. 결제/상품 관련 결측치 0으로 처리
fill_cols = [
    'payment_value',
    'payment_count',
    'payment_installments_mean',
    'payment_type_nunique',
    'item_count',
    'product_count',
    'seller_count',
    'product_price_sum',
    'freight_value_sum'
]

base_delivered[fill_cols] = base_delivered[fill_cols].fillna(0)

base_delivered[fill_cols].isnull().sum()

,0
payment_value,0
payment_count,0
payment_installments_mean,0
payment_type_nunique,0
item_count,0
product_count,0
seller_count,0
product_price_sum,0
freight_value_sum,0


- 결제/상품 정보가 없는 주문 → 계산 가능한 숫자 0으로 바꿔두기

In [ ]:
# 10. 기준일 설정
snapshot_date = base_delivered['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

snapshot_date

Timestamp('2018-08-30 15:00:37')

RFM
- 고객의 구매 행동을 3가지 숫자로 요약하는 방법
- RFM = Recency + Frequency + Monetary

| 항목               | 뜻              | 설명      |
| ---------------- | -------------- | ----------- |
| **R: Recency**   | 마지막 구매 후 지난 기간 | 얼마나 최근에 샀는가 |
| **F: Frequency** | 구매 횟수          | 얼마나 자주 샀는가  |
| **M: Monetary**  | 총 결제 금액        | 얼마나 많이 썼는가  |


- Recency를 계산하기 위한 기준 날짜를 정하는 단계
  - 데이터 안에서 가장 마지막 주문일을 찾고,
  - 거기에 하루를 더해서 기준일로 잡음
  > 가장 마지막 날에 구매한 고객의 Recency가 0일이 아니라 1일이 되도록 하기 위해서

In [ ]:
# 11. 고객별 RFM + 기본 구매 피처 생성
customer_features = base_delivered.groupby('customer_unique_id').agg(
    first_purchase_date=('order_purchase_timestamp', 'min'),
    last_purchase_date=('order_purchase_timestamp', 'max'),
    frequency=('order_id', 'nunique'),
    monetary=('payment_value', 'sum'),
    avg_payment_value=('payment_value', 'mean'),
    total_items=('item_count', 'sum'),
    avg_items_per_order=('item_count', 'mean'),
    total_product_price=('product_price_sum', 'sum'),
    total_freight_value=('freight_value_sum', 'sum'),
    unique_products=('product_count', 'sum'),
    unique_sellers=('seller_count', 'sum')
).reset_index()

customer_features.head()

,customer_unique_id,first_purchase_date,last_purchase_date,frequency,monetary,avg_payment_value,total_items,avg_items_per_order,total_product_price,total_freight_value,unique_products,unique_sellers
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05-10 10:56:27,1,141.90,141.90,1.0,1.0,129.90,12.00,1.0,1.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05-07 11:11:27,1,27.19,27.19,1.0,1.0,18.90,8.29,1.0,1.0
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-03-10 21:05:03,1,86.22,86.22,1.0,1.0,69.00,17.22,1.0,1.0
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-10-12 20:29:41,1,43.62,43.62,1.0,1.0,25.99,17.63,1.0,1.0
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11-14 19:45:42,1,196.89,196.89,1.0,1.0,180.00,16.89,1.0,1.0


- 주문 단위였던 base_delivered를 고객 단위로 바꾸는 작업

< 각 항목의 의미 >

| 코드                                                        | 의미              |
| --------------------------------------------------------- | --------------- |
| `first_purchase_date=('order_purchase_timestamp', 'min')` | 고객의 첫 구매일       |
| `last_purchase_date=('order_purchase_timestamp', 'max')`  | 고객의 마지막 구매일     |
| `frequency=('order_id', 'nunique')`                       | 고객의 총 주문 횟수     |
| `monetary=('payment_value', 'sum')`                       | 고객의 총 결제 금액     |
| `avg_payment_value=('payment_value', 'mean')`             | 고객의 평균 주문 결제 금액 |
| `total_items=('item_count', 'sum')`                       | 고객이 산 총 상품 수    |
| `avg_items_per_order=('item_count', 'mean')`              | 주문당 평균 상품 수     |
| `total_product_price=('product_price_sum', 'sum')`        | 총 상품 가격         |
| `total_freight_value=('freight_value_sum', 'sum')`        | 총 배송비           |
| `unique_products=('product_count', 'sum')`                | 구매한 상품 종류 수의 합  |
| `unique_sellers=('seller_count', 'sum')`                  | 구매한 판매자 수의 합    |


- 위 변수들은 LTV나 Churn 예측에서 고객의 구매 패턴을 설명하는 데 도움이 될 수 있어서 넣은 것
- 나중에 필요에 따라 삭제하거나 추가해서 모델링 해야함

In [ ]:
# 12. Recency, Tenure 계산
customer_features['recency'] = (
    snapshot_date - customer_features['last_purchase_date']
).dt.days

customer_features['tenure'] = (
    customer_features['last_purchase_date'] - customer_features['first_purchase_date']
).dt.days

customer_features.head()

,customer_unique_id,first_purchase_date,last_purchase_date,frequency,monetary,avg_payment_value,total_items,avg_items_per_order,total_product_price,total_freight_value,unique_products,unique_sellers,recency,tenure
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05-10 10:56:27,1,141.90,141.90,1.0,1.0,129.90,12.00,1.0,1.0,112,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05-07 11:11:27,1,27.19,27.19,1.0,1.0,18.90,8.29,1.0,1.0,115,0
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-03-10 21:05:03,1,86.22,86.22,1.0,1.0,69.00,17.22,1.0,1.0,537,0
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-10-12 20:29:41,1,43.62,43.62,1.0,1.0,25.99,17.63,1.0,1.0,321,0
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11-14 19:45:42,1,196.89,196.89,1.0,1.0,180.00,16.89,1.0,1.0,288,0


In [ ]:
customer_features.shape

(93358, 14)

In [ ]:
customer_features.describe()

,first_purchase_date,last_purchase_date,frequency,monetary,avg_payment_value,total_items,avg_items_per_order,total_product_price,total_freight_value,unique_products,unique_sellers,recency,tenure
count,93358,93358,93358.000000,93358.000000,93358.000000,93358.000000,93358.000000,93358.000000,93358.000000,93358.000000,93358.000000,93358.000000,93358.000000
mean,2018-01-01 12:18:32.380503296,2018-01-04 03:47:36.045363200,1.033420,165.197003,160.314930,1.180370,1.139531,141.621480,23.546730,1.073245,1.047784,237.941773,2.634032
min,2016-09-15 12:16:38,2016-09-15 12:16:38,1.000000,0.000000,0.000000,1.000000,1.000000,0.850000,0.000000,1.000000,1.000000,1.000000,0.000000
25%,2017-09-13 15:41:22.500000,2017-09-17 18:25:15,1.000000,63.052500,62.370000,1.000000,1.000000,47.650000,14.070000,1.000000,1.000000,114.000000,0.000000
50%,2018-01-20 06:48:07,2018-01-23 00:12:12.500000,1.000000,107.780000,105.630000,1.000000,1.000000,89.730000,17.600000,1.000000,1.000000,219.000000,0.000000
75%,2018-05-05 14:13:38.249999872,2018-05-07 17:22:34.750000128,1.000000,182.557500,176.650000,1.000000,1.000000,154.737500,25.520000,1.000000,1.000000,346.000000,0.000000
max,2018-08-29 15:00:37,2018-08-29 15:00:37,15.000000,13664.080000,13664.080000,24.000000,21.000000,13440.000000,1794.960000,15.000000,15.000000,714.000000,633.000000
std,NaN,NaN,0.209097,226.314012,219.571513,0.620857,0.527075,215.694014,22.780318,0.328448,0.250022,152.591453,24.955822


In [ ]:
customer_features.to_csv(
    '/content/drive/MyDrive/ESAA_OB_pj/customer/customer_features_baseline.csv',
    index=False
)

**< 고객 데이터셋 제작 과정 요약 >**

Olist 데이터 중 customers, orders, order_items, order_payments 데이터를 활용하여 분석 및 모델링에 사용할 수 있는 베이스라인 데이터셋 제작

전체 과정은 크게 세 단계로 구성

**1. 주문 단위 통합 데이터셋 base 생성**

  - orders와 customers는 공통 컬럼인 customer_id 기준으로 병합
  - payments와 items는 order_id 기준으로 병합하되, 한 주문에 여러 결제 내역이나 여러 상품이 존재할 수 있으므로 바로 병합하지 않고 먼저 order_id별로 집계 후 병합
  - 단순 병합 시 주문 1건이 여러 행으로 늘어나 고객별 구매 횟수나 결제금액 계산이 왜곡될 수 있기 때문
  - 최종적으로 base는 주문 1건당 1행인 통합 데이터셋이며, 주문 정보, 고객 정보, 결제 집계 정보, 상품 집계 정보, 날짜 정보 포함

**2. 이를 바탕으로 한 고객 단위 피처 데이터셋 customer_features 생성**

  - 구매일, 승인일, 배송일 등 날짜 관련 컬럼은 이후 Recency, Tenure 계산을 위해 datetime 형식으로 변환
  - 고객별 피처 생성 시 실제 구매가 완료된 주문을 기준으로 보기 위해 order_status == 'delivered'인 주문만 사용
  - 결제 및 상품 관련 컬럼에 일부 결측치가 존재하므로, 고객별 집계 과정에서 문제가 생기지 않도록 해당 값들을 0으로 처리

**3. 고객 단위 피처 데이터셋 생성**

  - base_delivered를 customer_unique_id 기준으로 집계하여 고객 1명당 1행인 customer_features 데이터셋 생성
  - 해당 데이터셋에는 RFM 관련 변수와 구매 행동을 설명하는 추가 변수 포함


> 최종 데이터

| 데이터명         | 변수명                 | 단위        | 설명                                                      |
| ------------ | ------------------- | --------- | ------------------------------------------------------- |
| 주문 단위 통합 데이터 | `base`              | 주문 1건당 1행 | customers, orders, payments, items를 병합한 주문 단위 베이스라인 데이터 |
| 고객 단위 피처 데이터 | `customer_features` | 고객 1명당 1행 | RFM 및 구매 행동 피처를 포함한 모델링용 고객 데이터                         |


***이후 회의 방향성***

1. 재구매 고객 비율 3%밖에 안 됨 → “고객 이탈 예측”으로 방향성 잡으면 고려사항 많아질 듯.
예) 불균형 처리(SMOTE 등…) 해서 고객 이탈 예측 먼저 해보기
2. “셀러 성공 예측”으로 메인 방향 변경하는 건?! → 이후 셀러 시뮬레이터 생성(”당신의 창업 얼마나 성공할 수 있을까요?”를 주제로, 학회원들 창업 희망 요소 입력하면 성공률 보여주기)
3. 이때 타겟변수를 재구매 유도율에서 변경!
- **평균 리뷰 평점** (회귀)
- **셀러 생존/이탈 여부** (분류)